In [1]:
import pandas as pd

In [2]:
df_listings_data = pd.read_csv("../data/London/extracted/listingsdata.csv")
df_calendar = pd.read_csv("../data/London/extracted/calendar.csv")
df_reviews_data = pd.read_csv("../data/London/extracted/reviewsdata.csv")
df_listings_info = pd.read_csv("../data/London/extracted/listings.csv")
df_reviews_info = pd.read_csv("../data/London/extracted/reviews.csv")
df_neighbourhoods = pd.read_csv("../data/London/extracted/neighbourhoods.csv")

In [ ]:
df_listings_data["price_clean"] = pd.to_numeric(  # covert to numeric in listings data
    df_listings_data["price"]
        .replace(r"[\$,]", "", regex=True),
    errors="coerce"
)

In [4]:
df_calendar["price_clean"] = pd.to_numeric(  # same in calender
    df_calendar["price"]
        .replace(r"[\$,]", "", regex=True),
    errors="coerce"
)

df_calendar["adjusted_price_clean"] = pd.to_numeric(
    df_calendar["adjusted_price"]
        .replace(r"[\$,]", "", regex=True),
    errors="coerce"
)

In [5]:
df_listings_info["price_clean"] = pd.to_numeric(   # same in listings info
    df_listings_info["price"]
        .replace(r"[\$,]", "", regex=True),
    errors="coerce"
)

In [ ]:
print(df_listings_data["price_clean"].dtype)  # quick validation
print(df_calendar["price_clean"].dtype)

float64
float64


## Parse and standardization

In [7]:
date_columns = [          # converting all date columns in listings data
    "last_scraped",
    "host_since",
    "calendar_last_scraped",
    "first_review",
    "last_review"
]

for col in date_columns:

    df_listings_data[col] = pd.to_datetime(
        df_listings_data[col],
        errors="coerce"
    )

In [ ]:
df_calendar["date"] = pd.to_datetime(   # same in calendar
    df_calendar["date"],
    errors="coerce"
)

In [9]:
df_reviews_data["date"] = pd.to_datetime(   # same in others
    df_reviews_data["date"],
    errors="coerce"
)

df_reviews_info["date"] = pd.to_datetime(
    df_reviews_info["date"],
    errors="coerce"
)

df_listings_info["last_review"] = pd.to_datetime(
    df_listings_info["last_review"],
    errors="coerce"
)

In [10]:
df_reviews_data["date"].dtype   # quick check

dtype('<M8[us]')

## Normalizing Free Text Fields

In [13]:
# exploring unique values

df_listings_data["property_type"].value_counts()


property_type
Entire rental unit             41215
Private room in rental unit    14464
Private room in home           11704
Entire home                     9120
Entire condo                    8250
                               ...  
Lighthouse                         1
Private room in resort             1
Shared room in loft                1
Private room in barn               1
Shared room in guest suite         1
Name: count, Length: 91, dtype: int64

In [14]:
df_listings_data["room_type"].value_counts()

room_type
Entire home/apt    62907
Private room       33643
Shared room          212
Hotel room           109
Name: count, dtype: int64

In [15]:
df_listings_data["property_type_clean"] = (  # turning to consistent format
    df_listings_data["property_type"]
        .str.strip()
        .str.lower()
)

In [16]:
df_listings_data["room_type_clean"] = (
    df_listings_data["room_type"]
        .str.strip()
        .str.lower()
)

## Handling Missing values

# dropping fully empty columns with reason

In [17]:
columns_to_drop = [
    "calendar_updated",
    "neighbourhood_group_cleansed"
]

df_listings_data.drop(
    columns=columns_to_drop,
    inplace=True
)

In [18]:
df_listings_info.drop(
    columns=["neighbourhood_group"],
    inplace=True
)

df_neighbourhoods.drop(
    columns=["neighbourhood_group"],
    inplace=True
)

# handling other

In [19]:
numeric_impute = [
    "beds",
    "bathrooms"
]

for col in numeric_impute:
    df_listings_data[col] = df_listings_data[col].fillna(
        df_listings_data[col].median()
    )

In [20]:
df_listings_data["host_response_time"] = (
    df_listings_data["host_response_time"]
    .fillna("Not Available")
)

df_listings_data["host_response_rate"] = (
    df_listings_data["host_response_rate"]
    .fillna("Not Available")
)

df_listings_data["host_acceptance_rate"] = (
    df_listings_data["host_acceptance_rate"]
    .fillna("Not Available")
)

In [ ]:
df_calendar[["price","adjusted_price"]].head()  # quick checks

,price,adjusted_price
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


In [22]:
df_calendar[["price","adjusted_price"]].sample(10)

,price,adjusted_price
9961346,NaN,NaN
18067195,NaN,NaN
21765148,NaN,NaN
24181324,NaN,NaN
30685068,NaN,NaN
22824745,NaN,NaN
30052839,NaN,NaN
10254172,NaN,NaN
30957240,NaN,NaN
29241565,NaN,NaN


In [ ]:
columns_to_drop1 = ["price","adjusted_price"] # dropped after investigation

df_calendar.drop(
    columns=columns_to_drop1,
    inplace=True
)

## Standardize Neighbourhood Names

In [24]:
df_listings_data["neighbourhood_cleansed"].unique()[:20]  # quick check

<StringArray>
[             'Islington', 'Kensington and Chelsea',            'Westminster',
             'Wandsworth',          'Tower Hamlets',   'Richmond upon Thames',
               'Haringey', 'Hammersmith and Fulham',              'Southwark',
                 'Barnet',               'Hounslow',         'Waltham Forest',
                  'Brent',                 'Camden',                'Hackney',
                 'Merton',                'Croydon',                'Lambeth',
               'Havering',              'Greenwich']
Length: 20, dtype: str

In [25]:
df_neighbourhoods["neighbourhood"].unique()

<StringArray>
[  'Barking and Dagenham',                 'Barnet',                 'Bexley',
                  'Brent',                'Bromley',                 'Camden',
         'City of London',                'Croydon',                 'Ealing',
                'Enfield',              'Greenwich',                'Hackney',
 'Hammersmith and Fulham',               'Haringey',                 'Harrow',
               'Havering',             'Hillingdon',               'Hounslow',
              'Islington', 'Kensington and Chelsea',   'Kingston upon Thames',
                'Lambeth',               'Lewisham',                 'Merton',
                 'Newham',              'Redbridge',   'Richmond upon Thames',
              'Southwark',                 'Sutton',          'Tower Hamlets',
         'Waltham Forest',             'Wandsworth',            'Westminster']
Length: 33, dtype: str

In [26]:
df_listings_data["neighbourhood_cleansed"] = (   # standardize
    df_listings_data["neighbourhood_cleansed"]
    .str.strip()
    .str.title()
)

df_neighbourhoods["neighbourhood"] = (
    df_neighbourhoods["neighbourhood"]
    .str.strip()
    .str.title()
)

In [27]:
listings_neighbourhoods = set(     # verifying inconsistancies
    df_listings_data["neighbourhood_cleansed"].dropna()
)

reference_neighbourhoods = set(
    df_neighbourhoods["neighbourhood"].dropna()
)

listings_neighbourhoods - reference_neighbourhoods

set()

In [ ]:
df_listings_data["host_location"] = (  # standardizing city names
    df_listings_data["host_location"]
    .str.strip()
    .str.title()
)

## Standardizing Coordinate Precision

In [30]:
df_listings_data["latitude"] = (
    df_listings_data["latitude"]
    .round(6)
)

df_listings_data["longitude"] = (
    df_listings_data["longitude"]
    .round(6)
)

In [31]:
from pathlib import Path

cleaned_dir = Path("../data/London/cleaned")
cleaned_dir.mkdir(parents=True, exist_ok=True)

In [32]:
df_listings_data.to_csv(
    cleaned_dir / "listingsdata_clean.csv",
    index=False
)

df_calendar.to_csv(
    cleaned_dir / "calendar_clean.csv",
    index=False
)

df_reviews_data.to_csv(
    cleaned_dir / "reviewsdata_clean.csv",
    index=False
)

df_listings_info.to_csv(
    cleaned_dir / "listingsinfo_clean.csv",
    index=False
)

df_reviews_info.to_csv(
    cleaned_dir / "reviewsinfo_clean.csv",
    index=False
)

df_neighbourhoods.to_csv(
    cleaned_dir / "neighbourhoods_clean.csv",
    index=False
)